In [ ]:
import pandas as pd
from datetime import datetime
import os
from dotenv import load_dotenv
from openai import OpenAI
import json
from collections import defaultdict
import random
from typing import List, Dict, Any, Tuple
import ast
from collections import defaultdict

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL")
client = OpenAI(api_key=OPENAI_API_KEY)
model = os.getenv("OPENAI_MODEL", "gpt-5-mini")

In [ ]:
# Load the original persona JSONL data
with open("data/페르소나 가중치 변환.jsonl", "r", encoding="utf-8") as f:
    personas = [json.loads(line) for line in f]
    
with open("data/product_info.json", "r", encoding="utf-8") as f:
    product_info_list = json.load(f)

In [ ]:
CLUSTER_CATALOG = {
    0: {"label": "실속형 미식가", "description": "편리성을 중시하면서도 새로운 맛과 제품을 시도하는 데 적극적인 소비자 그룹입니다. 가격에 민감하기보다는 효율성과 맛을 동시에 추구합니다."},
    1: {"label": "건강 추구형 소비자", "description": "가격이나 브랜드에 크게 구애받지 않고 건강을 최우선으로 고려하는 프리미엄 소비자 그룹입니다. 건강과 편리성을 모두 만족시키는 제품에 기꺼이 지갑을 엽니다."},
    3: {"label": "트렌드 주도형 소비자", "description": "자신의 취향과 경험을 중시하는 소비자 그룹입니다. 단순히 배를 채우는 것 이상의 가치를 추구하며, 새로운 제품을 가장 먼저 경험하고 공유하려는 경향이 강합니다."},
}

def enforce_cluster_meta(persona: Dict[str, Any]) -> Dict[str, Any]:
    """
    LLM이 생성한 persona에서 meta.cluster에 맞춰
    meta.label과 meta.description을 카탈로그 값으로 강제 세팅.
    """
    meta = persona.get("meta", {}).get("cluster")
    cluster = meta.get("cluster")

    # 방어적 캐스팅
    if isinstance(cluster, bool):
        cluster = int(cluster)
    if isinstance(cluster, (int, float)):
        cluster = int(cluster)
    else:
        raise ValueError("meta.cluster must be an integer.")

    if cluster not in CLUSTER_CATALOG:
        raise ValueError(f"Unknown cluster: {cluster}")

    meta["label"] = CLUSTER_CATALOG[cluster]["label"]
    meta["description"] = CLUSTER_CATALOG[cluster]["description"]
    return persona

def cluster_catalog_block() -> str:
    lines = ["[Cluster Catalog: DO NOT DEVIATE]"]
    for k, v in CLUSTER_CATALOG.items():
        lines.append(f"{k}:")
        lines.append(f"  label: \"{v['label']}\"")
        lines.append(f"  description: \"{v['description']}\"")
    return "\n".join(lines)

def meta_constraints_block() -> str:
    return (
        "[Meta Constraints]\n"
        "- \"meta.cluster\"가 정해지면, \"meta.label\"과 \"meta.description\"은 반드시 위 Catalog의 동일한 값을 그대로 복사한다.\n"
        "- 재해석·변형·요약 금지. 오타 금지."
    )

def operation_tips_block() -> str:
    return (
        "[Operation Tips]\n"
        "1) few-shot 안에도 3개의 서로 다른 cluster를 포함해, cluster가 바뀌면 label/description도 달라짐을 암시적으로 학습시킨다.\n"
        "2) 서버 보정(enforce_cluster_meta)을 항상 거친다. 프롬프트만 믿지 않는다.\n"
        "3) 클러스터 설명을 업데이트할 때는 Catalog 한 곳만 수정해 전체 파이프라인의 일관성을 유지한다.\n"
        "4) 예시 및 생성 결과에는 연령대(20대, 30대, 40대, 50대, 60대 이상), 성별, 직업의 다양성이 반영되도록 주의한다."
    )

def sample_with_demographic_diversity(personas: List[Dict[str, Any]], k: int = 10) -> List[Dict[str, Any]]:
    """
    cluster, age(value), gender(value)의 다양성을 우선 보장하며 few-shot용으로 샘플링
    """
    seen_keys = set()
    selected = []

    random.shuffle(personas)

    for p in personas:
        cluster = p.get("meta", {}).get("cluster")
        age = p.get("attributes", {}).get("age", {}).get("value")
        gender = p.get("attributes", {}).get("gender", {}).get("value")

        key = (cluster, age, gender)

        if key not in seen_keys:
            selected.append(p)
            seen_keys.add(key)

        if len(selected) >= k:
            break

    # 부족하면 나머지는 랜덤 보충
    if len(selected) < k:
        pool = [p for p in personas if p not in selected]
        random.shuffle(pool)
        selected.extend(pool[: k - len(selected)])

    return selected[:k]

In [ ]:
def flatten_persona_for_fewshot(p: dict) -> dict:
    return {
        "persona_key": p["persona_key"],
        "attributes": {k: v["value"] for k, v in p["attributes"].items()},
        "meta": p["meta"]
    }

def build_fewshot_block(personas: List[Dict[str, Any]], k=10) -> str:
    examples = sample_diverse_examples(personas, k)
    block = "다음은 소비자 페르소나 예시입니다:\n\n"
    for p in examples:
        p = enforce_cluster_meta(p)
        flat = {
            "persona_key": p["persona_key"],
            "attributes": {k: v["value"] for k, v in p["attributes"].items()},
            "meta": p["meta"]
        }
        block += "```json\n" + json.dumps(flat, ensure_ascii=False, indent=2) + "\n```\n\n"
    return block

In [ ]:
def build_conditioned_prompt(product_name: str, persona_index: int, few_shot_block: str) -> str:
    
    return f"""{cluster_catalog_block()}

{meta_constraints_block()}

{operation_tips_block()}

{few_shot_block}
이제 아래 조건을 만족하는 새로운 페르소나를 생성해주세요.

제품 ID: {product_name}
페르소나 번호: {product_name}_{persona_index}

[속성 예시값]
- gender: "남자", "여자"
- age: "20대", "30대", "40대", "50대", "60대 이상"
- job: "관리자", "군인", "기능원 및 관련 기능 종사자", "농림어업 숙련 종사자", "단순노무 종사자", "사무 종사자", "서비스 종사자",
        "전문가 및 관련 종사자", "판매 종사자", "장치·기계 조작 및 조립 종사자", "주부", "취업 준비 중", "학생"
- education: "고졸(대학 재학 포함)", "대학교 졸업(전문대졸/대학원생 포함)", "대학원 졸업 이상", "중졸 이하"
- region: "강원도", "경기도", "경상남도", "경상북도", "광주광역시", "대구광역시", "대전광역시", "서울특별시", "세종특별자치시",
        "울산광역시", "인천광역시", "전라남도", "전라북도", "제주특별자치도", "충청남도", "충청북도"
- household: "1인 가구", "1세대가족", "2세대가족"
- marriage: "기혼", "미혼(사별, 이혼 포함)"
- income_status: "맞벌이 하지 않음", "맞벌이 함"
- income_month : "100만원 미만", "100-200만원 미만", "200-300만원 미만", "300-400만원 미만", "400-500만원 미만", "500-600만원 미만",
        "600-700만원 미만", "700-800만원 미만", "800-900만원 미만", "900-1000만원 미만", "1,000만원 이상"
- brand_loyalty_scaled: 0.0 ~ 1.0 실수값
- cooking_convenience_scaled: 0.0 ~ 1.0 실수값
- health_orientation_scaled: 0.0 ~ 1.0 실수값
- hmr_preference_scaled: 0.0 ~ 1.0 실수값
- premium_orientation_scaled: 0.0 ~ 1.0 실수값
- price_sensitivity_scaled: 0.0 ~ 1.0 실수값
- variety_seeking_scaled: 0.0 ~ 1.0 실수값

[출력 형식 예시]
```json
{{
  "persona_key": "{product_name}_{persona_index}",
  "attributes": {{
    "gender": "여성",
    "age": "40대",
    "job": "전문직",
    "education": "대학교 졸업(전문대졸/대학원생 포함)",
    "region": "서울특별시",
    "household": "2인 가구",
    "marriage": "기혼",
    "income_status": "맞벌이 함"
    "income_month": 500-600만원 미만
    "brand_loyalty_scaled": 0.6,
    "cooking_convenience_scaled": 0.7,
    "health_orientation_scaled": 0.8,
    "hmr_preference_scaled": 0.95,
    "premium_orientation_scaled": 0.4,
    "price_sensitivity_scaled": 0.2,
    "variety_seeking_scaled": 0.5
  }},
  "meta": {{
    "cluster": 1,
    "label": "건강 추구형 소비자",
    "description": "가격이나 브랜드에 크게 구애받지 않고 건강을 최우선으로 고려하는 프리미엄 소비자 그룹입니다. 건강과 편리성을 모두 만족시키는 제품에 기꺼이 지갑을 엽니다."
  }}
}}
```"""

In [ ]:
def generate_monthly_prediction(product_name: str, launch_date: str, base_value: float = 3.0) -> Dict[str, Any]:
    launch = datetime.strptime(launch_date, "%Y.%m.%d")
    start = datetime(2024, 7, 1)

    row = {"product_name": product_name}
    for i in range(12):
        target = datetime(start.year + (start.month + i - 1) // 12, (start.month + i - 1) % 12 + 1, 1)
        col = f"months_since_launch_{i+1}"
        row[col] = 0 if target < launch else round(base_value + i * 0.1, 2)
    return row

In [ ]:
# 디렉토리 생성
os.makedirs("data/outputs/llm_outputs", exist_ok=True)
os.makedirs("data/outputs", exist_ok=True)

monthly_rows = []

for idx, product in enumerate(product_info_list):
    product_name = product["product_name"]
    release_date = product["release_date"]
    product_id = product_name  # 또는 f"product_{idx:02d}"
    
    few_shot_block = build_few_shot_block(personas, k=10)
    personas_output = []

    for i in range(20):
        prompt = build_conditioned_prompt(product_id, i, few_shot_block)

        try:
            # LLM 호출
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": "다음 조건을 만족하는 페르소나 정보를 JSON 형식(UTF-8, 인코딩된 따옴표 없음)으로 정확하게 생성하세요. 반드시 JSON만 출력하세요."},
                    {"role": "user", "content": prompt}
                ],
                temperature=1.2
            )

            # 응답 내용 정리
            content = response.choices[0].message.content.strip()
            if content.startswith("```json") or content.startswith("```"):
                content = content.split("```")[1].strip()
            if content.startswith("json"):
                content = content[4:].lstrip()

            print(f"[DEBUG] Raw response from LLM:\n{repr(content)}")

            # JSON 파싱 시도
            try:
                if "\\n" in content or '\\"' in content:
                    content = ast.literal_eval(content)  # 역직렬화
                persona_json = json.loads(content)     # JSON → dict

                persona_json = enforce_cluster_meta(persona_json)
                personas_output.append(persona_json)

            except Exception as json_err:
                print(f"[{product_name}] JSON 최종 파싱 실패: {json_err}")
                continue

        except Exception as llm_err:
            print(f"[{product_name}] persona 생성 실패: {llm_err}")
            continue

    # 페르소나 저장
    with open(f"data/outputs/llm_outputs/{product_name}_20_personas.json", "w", encoding="utf-8") as f:
        for p in personas_output:
            f.write(json.dumps(p, ensure_ascii=False) + "\n")

    # 월별 예측 저장
    monthly_rows.append(generate_monthly_prediction(product_name, release_date))

In [ ]:
monthly_df = pd.DataFrame(monthly_rows)
monthly_df.to_csv("data/outputs/final_monthly_prediction.csv", index=False, encoding="utf-8-sig")